# Benchmark Results Analysis

Loads audited model outputs, computes accuracy, per-difficulty breakdown, correlation with human solve rate, and cross-model comparison. Includes F1/Precision/Recall per difficulty tier.

## 1. Imports

In [ ]:
import re
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import pearsonr
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

WORKDIR = './audit_outputs'
GT_PATH = '/content/puzzle_samples.xlsx'
VALID_ANSWERS = {'A', 'B', 'C', 'D', 'E'}

## 2. Load Data

In [ ]:
# Load combined audited results
combined = pd.read_csv('/content/all_models_audited.csv')

# Load ground truth for Solved_Percentile
if GT_PATH.endswith('.xlsx'):
    gt = pd.read_excel(GT_PATH)
else:
    gt = pd.read_csv(GT_PATH)

def map_difficulty(p):
    if p < 30:   return 'Hard'
    elif p < 60: return 'Medium'
    else:        return 'Easy'

if 'Solved_Percentile' not in combined.columns:
    combined = combined.merge(
        gt[['Puzzle_Number', 'Question_Number', 'Solved_Percentile']],
        on=['Puzzle_Number', 'Question_Number'], how='left'
    )

combined['difficulty'] = combined['Solved_Percentile'].apply(map_difficulty)

# Only keep rows with valid audited answers for metric computation
valid = combined[combined['audited_answer'].isin(VALID_ANSWERS)].copy()

print(f'Total rows    : {len(combined)}')
print(f'Valid answers : {len(valid)}')
print(f'Models        : {sorted(combined["model_name"].unique())}')
print(f'\nDifficulty distribution:')
print(valid['difficulty'].value_counts().to_string())

## 3. Per-Model Accuracy Table

In [ ]:
DIFFICULTY_ORDER = ['Hard', 'Medium', 'Easy']

def model_metrics(g: pd.DataFrame) -> dict:
    """Accuracy + F1/Precision/Recall for a group."""
    y_true = g['GT_Answer']
    y_pred = g['audited_answer']
    acc    = (y_true == y_pred).mean()

    # Macro F1/P/R across A-E classes
    labels = list(VALID_ANSWERS)
    f1  = f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    pr  = precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    return {'acc': acc, 'f1': f1, 'precision': pr, 'recall': rec, 'n': len(g)}


rows = []
for model_name, g in valid.groupby('model_name'):
    m = model_metrics(g)
    row = {
        'model':     model_name,
        'n':         m['n'],
        'acc_%':     round(m['acc'] * 100, 1),
        'f1':        round(m['f1'],        3),
        'precision': round(m['precision'], 3),
        'recall':    round(m['recall'],    3),
    }
    for diff in DIFFICULTY_ORDER:
        sub = g[g['difficulty'] == diff]
        if len(sub):
            dm = model_metrics(sub)
            row[f'acc_{diff}_%'] = round(dm['acc'] * 100, 1)
            row[f'n_{diff}']     = dm['n']
        else:
            row[f'acc_{diff}_%'] = None
            row[f'n_{diff}']     = 0
    rows.append(row)

metrics_df = pd.DataFrame(rows).sort_values('acc_%', ascending=False)
metrics_df.to_csv(os.path.join(WORKDIR, 'model_metrics.csv'), index=False)

print('Cross-model metrics:')
print(metrics_df.to_string(index=False))

## 4. Accuracy vs Human Difficulty (Scatter + Correlation)

In [ ]:
# Use 4 bins matching the original viz but fixed to match difficulty mapping
BINS   = [0, 30, 60, 100]
LABELS = ['Hard\n(<30)', 'Medium\n(30-60)', 'Easy\n(≥60)']
BIN_CENTERS = [15, 45, 80]

models_list = sorted(valid['model_name'].unique())
ncols = 3
nrows = -(-len(models_list) // ncols)   # ceiling division

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for idx, model_name in enumerate(models_list):
    ax  = axes[idx]
    g   = valid[valid['model_name'] == model_name].copy()
    g['bin'] = pd.cut(g['Solved_Percentile'], bins=BINS, labels=LABELS, right=False)

    acc_by_bin = g.groupby('bin')['correct'].mean() * 100
    counts     = g.groupby('bin')['correct'].count()

    # Pearson on continuous percentile
    r, p = pearsonr(g['Solved_Percentile'], g['correct'].astype(int))
    p_str = 'p<0.001' if p < 0.001 else f'p={p:.3f}'

    ax.bar(BIN_CENTERS, acc_by_bin.reindex(LABELS).values,
           width=20, alpha=0.6, color=['#d62728','#ff7f0e','#2ca02c'],
           edgecolor='black', linewidth=0.5)
    ax.plot(BIN_CENTERS, acc_by_bin.reindex(LABELS).values,
            '-o', color='navy', linewidth=1.5, markersize=6, zorder=5)

    # Annotate counts
    for xc, label in zip(BIN_CENTERS, LABELS):
        n_bin = counts.reindex(LABELS).get(label, 0)
        ax.text(xc, 2, f'n={n_bin}', ha='center', va='bottom', fontsize=8, color='white', fontweight='bold')

    # Diagonal reference
    #ax.plot([0, 100], [0, 100], 'k--', alpha=0.25, label='Perfect alignment')

    ax.set_title(f'{model_name}\n(r={r:.3f}, {p_str})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Human Solved Percentile (%)', fontsize=9)
    ax.set_ylabel('Model Accuracy (%)', fontsize=9)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xticks([15, 45, 80])
    ax.set_xticklabels(['Hard\n<30', 'Medium\n30-60', 'Easy\n≥60'], fontsize=8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)

# Hide unused axes
for ax in axes[len(models_list):]:
    ax.set_visible(False)

plt.suptitle('Model Accuracy vs Human Difficulty', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'accuracy_vs_difficulty.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Cross-Model Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Overall accuracy ──────────────────────────────────────────────────────────
ax = axes[0]
order = metrics_df.sort_values('acc_%', ascending=True)
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(order)))
bars = ax.barh(order['model'], order['acc_%'], color=colors, edgecolor='black', linewidth=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlabel('Accuracy (%)', fontweight='bold')
ax.set_title('Overall Accuracy by Model', fontweight='bold')
ax.set_xlim(0, 110)
ax.axvline(x=20, color='grey', linestyle='--', alpha=0.4, label='Random (20%)')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)

# ── Accuracy by difficulty tier ───────────────────────────────────────────────
ax = axes[1]
x      = np.arange(len(metrics_df))
width  = 0.25
colors_diff = {'Hard': '#d62728', 'Medium': '#ff7f0e', 'Easy': '#2ca02c'}

for i, diff in enumerate(DIFFICULTY_ORDER):
    vals = metrics_df[f'acc_{diff}_%'].fillna(0)
    ax.bar(x + i * width, vals, width, label=diff,
           color=colors_diff[diff], alpha=0.75, edgecolor='black', linewidth=0.5)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_df['model'], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Accuracy (%)', fontweight='bold')
ax.set_title('Accuracy by Difficulty Tier', fontweight='bold')
ax.legend(title='Difficulty')
ax.set_ylim(0, 110)
ax.axhline(y=20, color='grey', linestyle='--', alpha=0.4, label='Random')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'cross_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. F1 / Precision / Recall Discussion

> For a 5-class MCQ task, **accuracy is the primary metric** since all classes are equally valid answer choices and there's no meaningful notion of false positives being worse than false negatives. Macro F1 is reported for completeness and to catch any systematic bias toward certain answer letters, but it should track closely with accuracy if the model isn't degenerate. If macro F1 diverges significantly from accuracy for a model, it suggests the model is over-predicting certain answer letters — worth investigating in the confusion matrix below.

## 7. Confusion Matrix per Model (answer letter bias check)

In [ ]:
fig, axes = plt.subplots(
    -(-len(models_list) // 4), min(4, len(models_list)),
    figsize=(4 * min(4, len(models_list)), 3.5 * (-(-len(models_list) // 4)))
)
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
labels = sorted(VALID_ANSWERS)

for idx, model_name in enumerate(models_list):
    ax = axes[idx]
    g  = valid[valid['model_name'] == model_name]
    cm = confusion_matrix(g['GT_Answer'], g['audited_answer'], labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(5)); ax.set_yticks(range(5))
    ax.set_xticklabels(labels); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(model_name, fontsize=9, fontweight='bold')

    for i in range(5):
        for j in range(5):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}',
                    ha='center', va='center', fontsize=7,
                    color='white' if cm_norm[i,j] > 0.5 else 'black')

for ax in axes[len(models_list):]:
    ax.set_visible(False)

plt.suptitle('Normalised Confusion Matrices (answer letter bias)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(WORKDIR, 'confusion_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Pearson Correlation Summary Table

In [ ]:
corr_rows = []
for model_name, g in valid.groupby('model_name'):
    r, p = pearsonr(g['Solved_Percentile'], g['correct'].astype(int))
    corr_rows.append({
        'model': model_name,
        'pearson_r': round(r, 3),
        'p_value':   round(p, 4),
        'significant': 'Yes' if p < 0.05 else 'No'
    })

corr_df = pd.DataFrame(corr_rows).sort_values('pearson_r', ascending=False)
corr_df.to_csv(os.path.join(WORKDIR, 'pearson_correlations.csv'), index=False)
print('Pearson r (accuracy vs human solve rate):')
print(corr_df.to_string(index=False))

I found it weird that many models show drop in medium level puzzles or does better in hard puzzles compared to easier ones.

2:39 AM
Claude responded: That's actually a known phenomenon in LLM benchmarking called non-monotonic difficulty scaling.
That's actually a known phenomenon in LLM benchmarking called non-monotonic difficulty scaling. A few explanations for what you're seeing:

Small n problem — your tiers are only ~33 puzzles each. A few lucky/unlucky guesses swing the accuracy significantly. The confidence intervals on 33 samples are wide enough that Hard and Medium differences may not be statistically meaningful.

Puzzle structure effect — "Hard" in your dataset means fewer humans solved it (P+ < 30), but that doesn't necessarily mean the logical structure is more complex for an LLM. Human difficulty correlates with things like time pressure, working memory load, and domain familiarity — none of which affect LLMs the same way. An LLM might find a "Hard" puzzle with a clean logical chain easier than a "Medium" puzzle with ambiguous wording.

Adaptation effect — your benchmark is adapted puzzles, not the originals. The adaptation quality varies by generator and the judge only checked logical equivalence, not difficulty preservation. A Hard puzzle that got adapted into a very clean domain might actually be easier for the model than the original.

Gemini specifically scoring highest on Hard is interesting — could be that Hard puzzles tend to have more explicit logical constraints which reasoning models handle well, while Medium puzzles have more ambiguous natural language.

Worth a brief note in the paper that difficulty monotonicity is not guaranteed for LLMs on adapted puzzles, and that the Pearson r is a more reliable signal than tier-by-tier comparisons at this sample size.